# Phase 4: TabNet (Google's Tabular Deep Learning)
TabNet mimics the sequential attention mechanism of decision trees, bringing the power of gradient boosting to deep learning. This allows it to learn sharp decision boundaries on tabular data while benefiting from neural network embeddings.

In [4]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
import torch
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Load NN Ready Data (Already Scaled & Log Transformed)
train_df = pd.read_parquet('../data/features/train_nn_ready.parquet', engine='fastparquet')
target_col = 'liquidity_stress_next_30d'
y = train_df[target_col].astype(np.float32).values
X = train_df.drop(columns=[target_col, 'ID'], errors='ignore').astype(np.float32).values

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (40000, 398), y shape: (40000,)


In [6]:
# 5-Fold CV for TabNet
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))

print("Training TabNet...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    clf = TabNetClassifier(
        n_d=32, n_a=32, n_steps=3, gamma=1.3,
        n_independent=2, n_shared=2, lambda_sparse=1e-3,
        optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        verbose=1
    )
    
    clf.fit(
        X_train=X_train, y_train=y_train,
        eval_set=[(X_val, y_val)],
        eval_name=['valid'],
        eval_metric=['logloss'],
        max_epochs=100 , patience=10,
        batch_size=1024, virtual_batch_size=128,
        num_workers=0,
        weights=1,
        drop_last=False
    )
    
    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]
    print(f"Fold {fold+1} Best LogLoss: {clf.best_cost}")
    
print(f"\nTabNet Final CV LogLoss: {log_loss(y, oof_preds):.4f}")
print(f"TabNet Final ROC-AUC: {roc_auc_score(y, oof_preds):.4f}")

Training TabNet...
epoch 0  | loss: 0.77015 | valid_logloss: 0.69775 |  0:00:02s
epoch 1  | loss: 0.67629 | valid_logloss: 0.67739 |  0:00:04s
epoch 2  | loss: 0.6322  | valid_logloss: 0.59576 |  0:00:06s
epoch 3  | loss: 0.60345 | valid_logloss: 0.57804 |  0:00:09s
epoch 4  | loss: 0.56649 | valid_logloss: 0.5628  |  0:00:11s
epoch 5  | loss: 0.53551 | valid_logloss: 0.54795 |  0:00:14s
epoch 6  | loss: 0.51401 | valid_logloss: 0.51504 |  0:00:16s
epoch 7  | loss: 0.47425 | valid_logloss: 0.51207 |  0:00:18s
epoch 8  | loss: 0.44149 | valid_logloss: 0.4857  |  0:00:20s
epoch 9  | loss: 0.412   | valid_logloss: 0.50792 |  0:00:23s
epoch 10 | loss: 0.3918  | valid_logloss: 0.48069 |  0:00:25s
epoch 11 | loss: 0.36026 | valid_logloss: 0.48479 |  0:00:27s
epoch 12 | loss: 0.33573 | valid_logloss: 0.43171 |  0:00:30s
epoch 13 | loss: 0.31172 | valid_logloss: 0.50932 |  0:00:32s
epoch 14 | loss: 0.29258 | valid_logloss: 0.47563 |  0:00:34s
epoch 15 | loss: 0.27644 | valid_logloss: 0.50428 |